# Data Preparation — GSE217498 (Mouse Stomach)

Load 10x Genomics matrices for WT mouse stomach samples from GSE217498.
Contains one corpus and one antrum sample.

**Source**: [GSE217498](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE217498)  
**Format**: 10x (barcodes.tsv.gz / genes.tsv.gz / matrix.mtx.gz)  
**Species**: Mouse  

**Note**: The deposited `.mtx.gz` files have incorrect headers (declared nnz doesn't match
actual entry count — extra data appended). This notebook reads only the declared entries,
bypassing scipy's `mmread` which rejects the files.

## 1. Setup & Imports

In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix, csr_matrix
import gzip

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(6, 4))

from importlib.metadata import version
print(f"Scanpy: {version('scanpy')}  |  AnnData: {version('anndata')}  |  NumPy: {np.__version__}  |  Pandas: {pd.__version__}")

Scanpy: 1.10.3  |  AnnData: 0.10.9  |  NumPy: 1.26.4  |  Pandas: 2.2.3


## 2. Project Configuration

In [2]:
PROJECT_NAME = 'gastric_antrum'
GEO_ID = 'GSE217498'

PROJECT_DIR = Path('..')
RAW_DIR = Path('RAW')
OUT_DIR = PROJECT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (prefix, sample_id, region)
SAMPLES = [
    ('GSM6720641_Wild_type_Corpus_', 'GSM6720641', 'corpus'),
    ('GSM6720642_Wild_type_Antrum_', 'GSM6720642', 'antrum'),
]

print(f"Project: {PROJECT_NAME}")
print(f"Dataset: {GEO_ID}")
print(f"Output:  {OUT_DIR}")
print(f"Samples: {len(SAMPLES)}")
for _, sid, region in SAMPLES:
    print(f"  {sid}  {region}")

Project: gastric_antrum
Dataset: GSE217498
Output:  ../data/processed
Samples: 2
  GSM6720641  corpus
  GSM6720642  antrum


## 3. Load Data

Custom 10x loader to handle malformed `.mtx.gz` headers.

In [3]:
def load_10x_malformed(raw_dir, prefix):
    """
    Read 10x files where the .mtx.gz has extra data appended beyond
    the declared nnz. Reads only the declared number of entries.
    """
    # Read barcodes and genes
    barcodes = pd.read_csv(raw_dir / f'{prefix}barcodes.tsv.gz',
                           header=None, sep='\t')[0].values
    genes_df = pd.read_csv(raw_dir / f'{prefix}genes.tsv.gz',
                           header=None, sep='\t')
    gene_names = genes_df[1].values

    # Parse matrix header to get declared dimensions
    mtx_path = raw_dir / f'{prefix}matrix.mtx.gz'
    with gzip.open(mtx_path, 'rt') as f:
        f.readline()  # %%MatrixMarket header
        dims = f.readline().strip().split()
        n_genes_mtx, n_cells_mtx, declared_nnz = int(dims[0]), int(dims[1]), int(dims[2])

    print(f"  Matrix header: {n_genes_mtx} genes x {n_cells_mtx} cells, {declared_nnz:,} entries")
    print(f"  Filtered files: {len(gene_names)} genes, {len(barcodes)} barcodes")

    # Read only the declared number of entries
    data = pd.read_csv(mtx_path, sep=r'\s+', skiprows=2, nrows=declared_nnz,
                       header=None, names=['row', 'col', 'val'],
                       dtype={'row': np.int32, 'col': np.int32, 'val': np.int32})

    # Build sparse matrix (MTX is 1-indexed)
    X = coo_matrix((data['val'].values, (data['row'].values - 1, data['col'].values - 1)),
                   shape=(n_genes_mtx, n_cells_mtx)).tocsr()

    # Subset to filtered genes/barcodes
    X_sub = X[:len(gene_names), :len(barcodes)]

    # Create AnnData (cells x genes)
    adata = ad.AnnData(
        X=X_sub.T.tocsr(),
        obs=pd.DataFrame(index=barcodes),
        var=pd.DataFrame(index=gene_names),
    )
    adata.var_names_make_unique()
    return adata

In [4]:
adatas = {}

for prefix, sample_id, region in SAMPLES:
    print(f"\nLoading {sample_id} ({region})...")
    adata = load_10x_malformed(RAW_DIR, prefix)

    adata.obs['sample_id'] = sample_id
    adata.obs['dataset'] = GEO_ID
    adata.obs['species'] = 'mouse'
    adata.obs['region'] = region
    adata.obs['condition'] = 'WT'

    for col in ['sample_id', 'dataset', 'species', 'region', 'condition']:
        adata.obs[col] = adata.obs[col].astype('category')

    adatas[sample_id] = adata
    print(f"  AnnData: {adata.shape}")

print(f"\nLoaded {len(adatas)} samples.")


Loading GSM6720641 (corpus)...
  Matrix header: 32285 genes x 8900 cells, 14,813,293 entries
  Filtered files: 19426 genes, 6951 barcodes


  AnnData: (6951, 19426)

Loading GSM6720642 (antrum)...
  Matrix header: 32285 genes x 12829 cells, 22,399,139 entries
  Filtered files: 19426 genes, 10813 barcodes


  AnnData: (10813, 19426)

Loaded 2 samples.


## 4. QC Metrics

In [5]:
for sample_id, adata in adatas.items():
    adata.var['mt'] = adata.var_names.str.startswith('mt-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

n_mt = adatas[list(adatas.keys())[0]].var['mt'].sum()
print(f"Mitochondrial genes detected: {n_mt} (prefix 'mt-')")

Mitochondrial genes detected: 0 (prefix 'mt-')


## 5. Summary

In [6]:
summary = []
for sample_id, adata in adatas.items():
    summary.append({
        'sample_id': sample_id,
        'region': adata.obs['region'].iloc[0],
        'n_cells': adata.n_obs,
        'n_genes': adata.n_vars,
        'median_genes': int(np.median(adata.obs['n_genes_by_counts'])),
        'median_counts': int(np.median(adata.obs['total_counts'])),
        'median_pct_mt': round(np.median(adata.obs['pct_counts_mt']), 2),
    })

summary_df = pd.DataFrame(summary)
print(f"{'='*80}")
print(f"GSE217498 Summary")
print(f"{'='*80}")
print(summary_df.to_string(index=False))
print(f"{'='*80}")
print(f"Total cells: {summary_df['n_cells'].sum():,}")

GSE217498 Summary
 sample_id region  n_cells  n_genes  median_genes  median_counts  median_pct_mt
GSM6720641 corpus     6951    19426           827           1410            0.0
GSM6720642 antrum    10813    19426          1007           1787            0.0
Total cells: 17,764


## 6. Save

In [7]:
# Save per-sample files
for sample_id, adata in adatas.items():
    out_path = OUT_DIR / f'{sample_id}_raw.h5ad'
    adata.write(out_path)
    print(f"Saved {out_path.name} ({adata.n_obs} cells)")

# Save combined file
combined = ad.concat(adatas, label='sample_key', join='outer', fill_value=0)
combined.obs_names_make_unique()
combined_path = OUT_DIR / f'{GEO_ID}_combined_raw.h5ad'
combined.write(combined_path)
print(f"\nSaved {combined_path.name} ({combined.n_obs} cells x {combined.n_vars} genes)")

Saved GSM6720641_raw.h5ad (6951 cells)


Saved GSM6720642_raw.h5ad (10813 cells)


/Users/zahra/miniconda3/envs/bioinfo/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")



Saved GSE217498_combined_raw.h5ad (17764 cells x 19426 genes)


In [8]:
# Verify
test = sc.read_h5ad(combined_path)
print(f"Verification: {combined_path.name}: {test.shape[0]} cells x {test.shape[1]} genes  OK")

Verification: GSE217498_combined_raw.h5ad: 17764 cells x 19426 genes  OK
